In [ ]:
!pip install git+https://github.com/begelb/latent_dynamics.git@paper

# Section 5.2.1 - Three-dimensional Leslie: a spurious attractor

## What this notebook shows

The three-generation **Leslie population model** (paper section 5.2), a
genuinely three-dimensional system, studied through a **two-dimensional** latent
model. This is the *cautionary tale*: the latent Morse graph comes out
**tristable** (three minimal nodes), but the true system has only two stable
period-four orbits -- one latent attractor corresponds to no attractor of the
real dynamics.

This is not just numerical noise: the semiconjugacy-error bound required by the
main theorem is *violated* at that node. A small training loss does **not** by
itself guarantee that latent attractors are real. (Example 2, in the next
notebook, is the success case.)

### How to run

Edit the **parameters** cell below, then *Run All*. Three modes:

| `MODE` | what it does | cost |
|--------|--------------|------|
| `"replay"` | re-render the paper's saved Morse graph and Morse sets | seconds |
| `"morse"` | recompute the Morse graph of the *saved* model at your `SUBDIV` | seconds-minutes |
| `"retrain"` | run the whole pipeline from scratch with your `OVERRIDES` | minutes-hours |

**Toy subdivisions are a qualitative preview.** Coarse CMGDB grids can merge
nearby recurrent sets and change the Morse graph; the paper figures use the
config's (finer) values. The paper value for this example is noted in the
parameters cell.

In [ ]:
# ===== PARAMETERS  (edit, then Run All) ====================================
MODE = "replay"            # "replay" | "morse" | "retrain"
SEED = None                # None -> the config's default seed
SUBDIV = (10, 14, 20)      # MODE="morse": (subdiv_init, subdiv_min, subdiv_max)
                           # paper value: (23, 23, 27)
OVERRIDES = {}             # MODE="retrain": config overrides (pydantic-validated), e.g.
                           #   {"training": {"epochs": 300}, "cmgdb": {"subdiv_max": 20}}
BOX_SCALE = "auto"         # Morse-set box size: "auto" | float | {label: float}
# ===========================================================================

In [ ]:
from latentdynamics.replay import load_experiment, retrain

REPLAY_CONFIG  = "leslie3d_example1_replay"
RETRAIN_CONFIG = "leslie3d_example1"

if MODE == "replay":
    exp = load_experiment(REPLAY_CONFIG, seed=SEED)
elif MODE == "morse":
    exp = load_experiment(REPLAY_CONFIG, seed=SEED).recompute_morse(subdiv=SUBDIV)
elif MODE == "retrain":
    exp = retrain(RETRAIN_CONFIG, seed=SEED, overrides=OVERRIDES)
else:
    raise ValueError(f"unknown MODE {MODE!r}")
exp

## Morse graph

Three minimal nodes -- one of them spurious.

In [ ]:
exp.show_morse_graph()

## Morse sets

In [ ]:
exp.show_morse_sets(box_scale=BOX_SCALE)

## Latent trajectories of the two 4-periodic orbits

Each true period-four orbit is encoded into the latent space and pushed forward
under the latent map. The orbit inside the spurious Morse set never settles --
that is the failure this example exposes. (2-D latent only; shown for the
replay/morse model.)

In [ ]:
from latentdynamics.cli.render import LESLIE3D_PERIODIC_PTS

exp.show_latent_trajectory(LESLIE3D_PERIODIC_PTS, steps=4)

## Run provenance

In [ ]:
exp.diagnostics()